# TESPy tutorial: Geothermal ORC

In this tutorial we build a tespy model step by step and explore a couple of 
the software's features. The plant uses 100 kg/s geothermal brine at 140 °C to
evaporate isopentane. An air cooled condenser rejects the waste heat.

## Step 1: imports, Network and units

First, import all necessary dependencies. The `Network` class is the container
for each model, the units specify the default unit system.

In [ ]:
from tespy.networks import Network
from tespy.components import (
    Source, Sink,
    Pump, Turbine,
    MovingBoundaryHeatExchanger,
    CycleCloser,
    PowerSink, PowerBus,
    Motor, Generator
)
from tespy.connections import Connection, PowerConnection

import matplotlib.pyplot as plt

In [ ]:
nw = Network()

nw.units.set_defaults(
    temperature="degC", pressure="bar", pressure_difference="bar",
    enthalpy="kJ/kg", mass_flow="kg/s", power="kW", heat="kW"
)

## Step 2: components

Next, we create the components of the cycle as shown in the flowsheet. The
`CycleCloser` is used to close a loop.

**Checkpoint 1**: open `checkpoints/checkpoint_1_components.ipynb`, run all
cells and continue from here

## Step 3: fluid connections

The `Connection` class instances are used to link an outlet of a component to
the inlet of a different one. They carry the fluid information and variables
for the solver: mass flow, pressure, enthalpy and fluid composition.

## Step 4: power connections

The `PowerConnection` class instances are used to represent non-material energy
flow between components, e.g. electrical and mechanical energy. They only
represent a single variable for the solver, i.e. the energy flow.

**Checkpoint 2**: open `checkpoints/checkpoint_2_connections.ipynb`, run all
cells and continue from here

## Step 5: parameters

To solve a model subject to the boundary conditions we need to make sure, every
mass flow, pressure, enthalpy, fluid composition and energy flow can be 
determined from our specifications. We implement the following boundary
conditions:

| where | specification | meaning |
| --- | --- | --- |
| `a1` | `fluid`, `T=140`, `p=10`, `m=100` | geothermal source conditions |
| `b1` | `fluid`, `x=1`, `T=105` | saturated vapour at turbine inlet |
| `b3` | `td_bubble=5` | 5 K subcooling after the condenser |
| `c1`, `c2` | `T=10` / `T=20` | ambient air heated from 10 to 20 °C |
| turbine, pump | `eta_s` | isentropic efficiencies |
| generator, motor | `eta` | power train losses |
| heat exchangers | `dp1=0`, `dp2=0` | no pressure losses |
| condenser, evaporator | `td_pinch` | minimum temperature difference between flows |


**Checkpoint 3**: open `checkpoints/checkpoint_3_parametrized.ipynb`, run all
cells and continue from here

## Step 6: solve

We solve the model by calling `nw.solve()`, `nw.print_results()` provides an
overview of the most important results.

### Results

We can also extract specific results or do some postprocessing:

In [ ]:
print(f"net power:           {e5.E.val:8.1f} kW")
print(f"heat input:          {abs(evaporator.Q.val):8.1f} kW")
print(f"thermal efficiency:  {e5.E.val / abs(evaporator.Q.val) * 100:8.2f} %")
print(f"working fluid flow:  {b1.m.val:8.2f} kg/s")
print(f"brine outlet:        {a2.T.val:8.2f} degC")

### Heat exchangers

We can plot Q-T diagrams to show the heat exchanger pinches

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, heatex in zip(axs, ["evaporator", "condenser"]):
    comp = nw.get_comp(heatex)
    ax.plot(comp.Q_sections.val, comp.T_hot_sections.val, "r-o", ms=3, label="hot")
    ax.plot(comp.Q_sections.val, comp.T_cold_sections.val, "b-o", ms=3, label="cold")
    ax.set_title(f"{heatex} (pinch = {comp.td_pinch.val:.2f} K)")
    ax.set_xlabel("cumulative heat duty Q [kW]")
    ax.set_ylabel("temperature T [degC]")
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()

## Step 7: Debugging

We can debug a model, in case failures happen during solving. For this, we
deliberately specify boundary conditions that lead to failures.

### Overdetermination of mass flow

We can specify an extra boundary condition, i.e. working fluid mass flow, and
try to solve again:

We can undo our wrong specification and instead specify a unsatisfyable boudary
condition, e.g. a pinch of 40 K. This leads to non-convergence, since the 
residual of the pinch equation can never be satisfied.

## Step 8: design and off-design

It is possible to run design and offdesign simulations. E.g. how large is the
evaporator to achieve a pinch of 8 K. The `UA` value is the result of that
boundary condition.

In offdesign simulation mode we invert that, i.e. the `UA` value might be a
fixed input and the pinch becomes a result. 

TESPy does not provide you with help on what to switch from design to
offdesign, it is entirely your knowledge about the system to make this
decision.

In [ ]:
print(f"design:     UA = {evaporator.UA.val / 1e3:7.1f} kW/K  (result)")
print(f"            pinch = {evaporator.td_pinch.val:5.2f} K  (specified)")

Now we run the simulation in offdesign mode in which the `UA` value of the
evaporator is fixed to its design point value. We can change the mass flow and
see how the pinch changes.

In [ ]:
print(f"off-design: UA = {evaporator.UA.val / 1e3:7.1f} kW/K  (specified)")
print(f"            pinch = {evaporator.td_pinch.val:5.2f} K  (result)")
print(f"            {a1.m.val} kg/s brine -> {e5.E.val:.1f} kW")

When reducing the mass flow, the pinch reduces as the same heat exchange area
is available as in the design simulation but less heat is transferred due to
the lower mass flow.

After this, we go back to the original value of `td_pinch` and run a design
simulation.

In [ ]:
a1.set_attr(m=100)

# the off-design run left its own result in td_pinch - a design
# parameter keeps whatever value it currently holds, so set it again
evaporator.set_attr(td_pinch=8)

nw.solve("design")

## Step 9: modeling flexibility

TESPy provides you with a lot of flexibility in modeling: You can implement
relations into your model, that are not available from the library. For this
the `UserDefinedEquation` is used. 

Suppose, you want to maximize the power output of the ORC plant. A rough
estimation for the optimal evaporation temperature can be given by a rule of
thumb: the optimum sits near the **geometric mean** of source and sink
temperature in Kelvin [1]:

$$T_\text{evaporation} \approx \sqrt{T_\text{source} \cdot
T_\text{sink}}$$

We can implement this equation in its residual form as an extra boundary
condition to replace the direct temperature specification at the evaporator
outlet.

In [ ]:
print(f"source                  {a1.T.val:7.2f} degC")
print(f"sink                    {b3.T.val:7.2f} degC")
print(f"evaporation (the rule)  {b1.T.val:7.2f} degC")
print()
print(f"net power               {e5.E.val:7.1f} kW   (was 2665.0 at 105 degC)")
print(f"thermal efficiency      {e5.E.val / abs(evaporator.Q.val) * 100:7.2f} %    (was 12.91)")

With the changed evaporation temperature we yield higher power output at a 
decreased efficiency. This is possible as we are able to remove more heat from
the brine flow. 

Next, with a small sensitivity analysis we can check, how good the rule of
thumb is. We remove the additional equation again and instead loop over 
different evaporation temperatures.

In [ ]:
print(" T_evap    P_net      eta_th")
for T, (P, eta) in results.items():
    mark = "  <- the rule" if T == 74.33 else ""
    print(f"{T:6.2f} {P:9.1f} kW {eta:7.2f} %{mark}")

best = max(results, key=lambda T: results[T][0])
print(f"\nmaximum power at {best} degC: {results[best][0]:.1f} kW")

The rule of thumb is quite close in terms of absolute power output to the
actual optimum.

## Reference

**[1]** Curzon, F. L. and Ahlborn, B. (1975). *Efficiency of a Carnot engine at
maximum power output.* American Journal of Physics 43(1), 22-24.
doi: [10.1119/1.10023](https://doi.org/10.1119/1.10023)